# Data Wrangling and Feature Engineering
This notebook cleans the course-provided SpaceX dataset, documents missing values, creates a binary landing-success target and encodes categorical predictors. It deliberately reports observed results rather than hard-coding expected counts.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(DATA_URL)
print("Initial shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))

In [ ]:
# Normalize column names only where useful; preserve original names for traceability.
df.columns = [c.strip() for c in df.columns]
required = ["FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Landing_Outcome"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    print("Dataset columns:", df.columns.tolist())
    raise KeyError(f"Expected columns not found: {missing_cols}")

In [ ]:
# Handle payload missing values using the median, which is less sensitive to outliers than the mean.
df["PayloadMass"] = pd.to_numeric(df["PayloadMass"], errors="coerce")
payload_median = df["PayloadMass"].median()
df["PayloadMass"] = df["PayloadMass"].fillna(payload_median)

# Define success from landing outcome labels. 'Success' outcomes are positive;
# failures, no-attempt and controlled/uncontrolled ocean outcomes are not counted as successful landings.
df["Class"] = df["Landing_Outcome"].astype(str).str.startswith("Success").astype(int)
print("Missing payload values after imputation:", int(df["PayloadMass"].isna().sum()))
display(df["Class"].value_counts(dropna=False).rename("count").to_frame())

In [ ]:
# Select useful predictors available in this dataset.
numeric_candidates = [c for c in ["FlightNumber", "PayloadMass"] if c in df.columns]
categorical_candidates = [c for c in ["Orbit", "LaunchSite", "LandingPad", "Serial", "Booster_Version"] if c in df.columns]
features = df[numeric_candidates + categorical_candidates].copy()
for col in numeric_candidates:
    features[col] = pd.to_numeric(features[col], errors="coerce")
    features[col] = features[col].fillna(features[col].median())
for col in categorical_candidates:
    features[col] = features[col].fillna("Unknown").astype(str)

X = pd.get_dummies(features, columns=categorical_candidates, drop_first=False, dtype=int)
y = df["Class"].astype(int)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())

In [ ]:
df.to_csv("spacex_cleaned.csv", index=False)
X.to_csv("spacex_features.csv", index=False)
y.to_frame("Class").to_csv("spacex_target.csv", index=False)
print("Saved spacex_cleaned.csv, spacex_features.csv, and spacex_target.csv")

## Data-wrangling notes
- Missing payload mass is imputed with the median.
- Landing outcomes beginning with `Success` are encoded as 1; other outcomes are encoded as 0.
- Categorical predictors are one-hot encoded.
- The feature matrix and target are saved separately for reproducibility.
- Review the dataset's outcome semantics before interpreting the target as a universal definition of landing success.